# RepairScopeX E02 — Context_Corruption
**New exploratory ALoDLM FP32 experiment · GPU Colab · checkpointed**

This is an independent study under `/MyDrive/RepairScopeX/E_STUDIES/E02`. It does not read or write NB29H/V1 trial stores. Frozen bank entries are embedded and come from calibration arithmetic tasks that have **not** been used by the existing V1, NB29G or NB29H plans (as of the input snapshots). Each problem has 3 paired prompt conditions and 2 common seeds: **72 new trial slots**.

Controls: paired seeds and common decoder; outcomes are **correct / definite_wrong / ambiguous**, never turn ambiguous into wrong. This is a *pilot*, not independent held-out model confirmation. Session checkpoints save immediately and skip completed trial IDs.

**Run:** GPU (T4, 12GiB+), Runtime → Run all. Each run adds ≤12 new trials or ≤30 minutes; launch again after Colab disconnects. Results export even if the study is incomplete. Download the `RSX_E02_RESULTS.zip` generated by the final cell.

In [ ]:
import os,base64,json
STUDY='E02'
ARMS=['no_hint', 'true_hint', 'false_hint']
PROBLEMS=json.loads(base64.b64decode('W3siY29sbGlzaW9uX3JldHJ5IjogMCwgImtpbmQiOiAiYXJpdGhtZXRpYyIsICJudW1iZXJzIjogW10sICJwcm9ibGVtX2ZpbmdlcnByaW50IjogImQ0YzEyZjMzOTA3MDA4OWVjZDE4YjkzMzdlMzAxYzUxMDNkNmU3ZDA0MGUzMjk2MTc3OWNiZGM3YjU5MmY3MTEiLCAicHJvYmxlbV9pZCI6ICJyc3gyOTpjYWxpYnJhdGlvbjphcml0aG1ldGljOjAwNzciLCAicHJvbXB0IjogIkEgc2hlbGYgaG9sZHMgNDQgYm9va3MuIEV4YWN0bHkgMTMgYm9va3MgYXJlIGJvcnJvd2VkLiBIb3cgbWFueSBib29rcyByZW1haW4/IEVuZCB3aXRoIEZJTkFMX0FOU1dFUjogPG51bWJlcj4uIiwgInNwbGl0IjogImNhbGlicmF0aW9uIiwgInRhcmdldCI6IDMxLjAsICJ0ZW1wbGF0ZSI6IDF9LCB7ImNvbGxpc2lvbl9yZXRyeSI6IDAsICJraW5kIjogImFyaXRobWV0aWMiLCAibnVtYmVycyI6IFtdLCAicHJvYmxlbV9maW5nZXJwcmludCI6ICIzYjM2NGZmMTNiNDI4MmMwZDEzMTIzMWIyY2I1NzExMzY1YjRmZWYxZTM5MDFmZGMzOThjMTIxMDZmYmU3ZjYzIiwgInByb2JsZW1faWQiOiAicnN4Mjk6Y2FsaWJyYXRpb246YXJpdGhtZXRpYzowMDc2IiwgInByb21wdCI6ICJBIGNvbGxlY3Rpb24gY29udGFpbnMgNTYgY29pbnMuIEEgZnJpZW5kIGFkZHMgMjggbW9yZSBjb2lucy4gSG93IG1hbnkgY29pbnMgYXJlIHRoZXJlIG5vdz8gRW5kIHdpdGggRklOQUxfQU5TV0VSOiA8bnVtYmVyPi4iLCAic3BsaXQiOiAiY2FsaWJyYXRpb24iLCAidGFyZ2V0IjogODQuMCwgInRlbXBsYXRlIjogMH0sIHsiY29sbGlzaW9uX3JldHJ5IjogMCwgImtpbmQiOiAiYXJpdGhtZXRpYyIsICJudW1iZXJzIjogW10sICJwcm9ibGVtX2ZpbmdlcnByaW50IjogIjYyNTk1ZDMzMTcwYjkwYTg5YzllM2I2YTdmNzEwMTA1OGI4MGM3NTJhMmNkNTU1ZGNjNGI1MTA3MWQ5ZDMxYjIiLCAicHJvYmxlbV9pZCI6ICJyc3gyOTpjYWxpYnJhdGlvbjphcml0aG1ldGljOjAwODQiLCAicHJvbXB0IjogIkEgY29sbGVjdGlvbiBjb250YWlucyAyOCBjb2lucy4gQSBmcmllbmQgYWRkcyAxOCBtb3JlIGNvaW5zLiBIb3cgbWFueSBjb2lucyBhcmUgdGhlcmUgbm93PyBFbmQgd2l0aCBGSU5BTF9BTlNXRVI6IDxudW1iZXI+LiIsICJzcGxpdCI6ICJjYWxpYnJhdGlvbiIsICJ0YXJnZXQiOiA0Ni4wLCAidGVtcGxhdGUiOiAwfSwgeyJjb2xsaXNpb25fcmV0cnkiOiAwLCAia2luZCI6ICJhcml0aG1ldGljIiwgIm51bWJlcnMiOiBbXSwgInByb2JsZW1fZmluZ2VycHJpbnQiOiAiNzIyOTU1NGEzMjEzYmZjMmJiNTk0NWMwMGU2Y2M0OWI4NTU0Nzk3NjRkYmIxMzc0OWIxMjQyN2ZhYWYyYzhiNyIsICJwcm9ibGVtX2lkIjogInJzeDI5OmNhbGlicmF0aW9uOmFyaXRobWV0aWM6MDA4MSIsICJwcm9tcHQiOiAiQSBzaGVsZiBob2xkcyA2MyBib29rcy4gRXhhY3RseSAyNyBib29rcyBhcmUgYm9ycm93ZWQuIEhvdyBtYW55IGJvb2tzIHJlbWFpbj8gRW5kIHdpdGggRklOQUxfQU5TV0VSOiA8bnVtYmVyPi4iLCAic3BsaXQiOiAiY2FsaWJyYXRpb24iLCAidGFyZ2V0IjogMzYuMCwgInRlbXBsYXRlIjogMX0sIHsiY29sbGlzaW9uX3JldHJ5IjogMCwgImtpbmQiOiAiYXJpdGhtZXRpYyIsICJudW1iZXJzIjogW10sICJwcm9ibGVtX2ZpbmdlcnByaW50IjogImM0OGYyOGEwMDM1NzQzMzEyMjJhOGQzMTEzZDIwY2UyYWRiMzg1ZWNhZjY4ZmQxM2FiODliNjc2Y2M1NjIzZjEiLCAicHJvYmxlbV9pZCI6ICJyc3gyOTpjYWxpYnJhdGlvbjphcml0aG1ldGljOjAwNTkiLCAicHJvbXB0IjogIkEgbnVtYmVyIG9mIHN0aWNrZXJzIGlzIGRpdmlkZWQgZXF1YWxseSBhbW9uZyA4IGNoaWxkcmVuLCBhbmQgZXZlcnkgY2hpbGQgcmVjZWl2ZXMgNDEgc3RpY2tlcnMuIEhvdyBtYW55IHN0aWNrZXJzIHdlcmUgdGhlcmUgb3JpZ2luYWxseT8gRW5kIHdpdGggRklOQUxfQU5TV0VSOiA8bnVtYmVyPi4iLCAic3BsaXQiOiAiY2FsaWJyYXRpb24iLCAidGFyZ2V0IjogMzI4LjAsICJ0ZW1wbGF0ZSI6IDN9LCB7ImNvbGxpc2lvbl9yZXRyeSI6IDAsICJraW5kIjogImFyaXRobWV0aWMiLCAibnVtYmVycyI6IFtdLCAicHJvYmxlbV9maW5nZXJwcmludCI6ICJkMmMyNTljOTI0ZjBiZTBkMDEyZWU3MTU4ZmFmYzBkNWI0ODUxZGIxODdiZTRkNjljZTY4MGQ4MWUxOTQ5ODBmIiwgInByb2JsZW1faWQiOiAicnN4Mjk6Y2FsaWJyYXRpb246YXJpdGhtZXRpYzowMDg2IiwgInByb21wdCI6ICJUaGVyZSBhcmUgMzggaWRlbnRpY2FsIGJveGVzLCBlYWNoIGNvbnRhaW5pbmcgMTMgcGVucy4gSG93IG1hbnkgcGVucyBhcmUgdGhlcmUgYWx0b2dldGhlcj8gRW5kIHdpdGggRklOQUxfQU5TV0VSOiA8bnVtYmVyPi4iLCAic3BsaXQiOiAiY2FsaWJyYXRpb24iLCAidGFyZ2V0IjogNDk0LjAsICJ0ZW1wbGF0ZSI6IDJ9LCB7ImNvbGxpc2lvbl9yZXRyeSI6IDAsICJraW5kIjogImFyaXRobWV0aWMiLCAibnVtYmVycyI6IFtdLCAicHJvYmxlbV9maW5nZXJwcmludCI6ICIzMTE4N2NjN2IxNGQ3OGZhMGU1Y2MzYTQzZmM4YTc0YmE5ZmQxM2JhM2FlM2MzZGE4M2ZkOTBlMDRjYzEzZjU4IiwgInByb2JsZW1faWQiOiAicnN4Mjk6Y2FsaWJyYXRpb246YXJpdGhtZXRpYzowMDU3IiwgInByb21wdCI6ICJBIHNoZWxmIGhvbGRzIDQxIGJvb2tzLiBFeGFjdGx5IDYgYm9va3MgYXJlIGJvcnJvd2VkLiBIb3cgbWFueSBib29rcyByZW1haW4/IEVuZCB3aXRoIEZJTkFMX0FOU1dFUjogPG51bWJlcj4uIiwgInNwbGl0IjogImNhbGlicmF0aW9uIiwgInRhcmdldCI6IDM1LjAsICJ0ZW1wbGF0ZSI6IDF9LCB7ImNvbGxpc2lvbl9yZXRyeSI6IDAsICJraW5kIjogImFyaXRobWV0aWMiLCAibnVtYmVycyI6IFtdLCAicHJvYmxlbV9maW5nZXJwcmludCI6ICIyOGYwMDI4NTMzMTViNjNiZjdkNzg5ZDJmYTU0ZDliMTA5NzMxODRlMDFlMmQxNmI3NzJmZTExMTgwYzlkYmQwIiwgInByb2JsZW1faWQiOiAicnN4Mjk6Y2FsaWJyYXRpb246YXJpdGhtZXRpYzowMDY2IiwgInByb21wdCI6ICJUaGVyZSBhcmUgOTggaWRlbnRpY2FsIGJveGVzLCBlYWNoIGNvbnRhaW5pbmcgNiBwZW5zLiBIb3cgbWFueSBwZW5zIGFyZSB0aGVyZSBhbHRvZ2V0aGVyPyBFbmQgd2l0aCBGSU5BTF9BTlNXRVI6IDxudW1iZXI+LiIsICJzcGxpdCI6ICJjYWxpYnJhdGlvbiIsICJ0YXJnZXQiOiA1ODguMCwgInRlbXBsYXRlIjogMn0sIHsiY29sbGlzaW9uX3JldHJ5IjogMCwgImtpbmQiOiAiYXJpdGhtZXRpYyIsICJudW1iZXJzIjogW10sICJwcm9ibGVtX2ZpbmdlcnByaW50IjogIjE0OGM5NmM0OTFlNTE0NTFhN2ZhNjI5MzBmZTQ1ZDVkNzM2N2ViYTc0ODAxMjdlMGU3YjQyMDlhYjMxM2UwYjkiLCAicHJvYmxlbV9pZCI6ICJyc3gyOTpjYWxpYnJhdGlvbjphcml0aG1ldGljOjAwNzQiLCAicHJvbXB0IjogIlRoZXJlIGFyZSAzNyBpZGVudGljYWwgYm94ZXMsIGVhY2ggY29udGFpbmluZyAyNSBwZW5zLiBIb3cgbWFueSBwZW5zIGFyZSB0aGVyZSBhbHRvZ2V0aGVyPyBFbmQgd2l0aCBGSU5BTF9BTlNXRVI6IDxudW1iZXI+LiIsICJzcGxpdCI6ICJjYWxpYnJhdGlvbiIsICJ0YXJnZXQiOiA5MjUuMCwgInRlbXBsYXRlIjogMn0sIHsiY29sbGlzaW9uX3JldHJ5IjogMCwgImtpbmQiOiAiYXJpdGhtZXRpYyIsICJudW1iZXJzIjogW10sICJwcm9ibGVtX2ZpbmdlcnByaW50IjogImIyNGNhODA2MDRmYThiMTE0ODgxNzUwOTgzNjg1ZDE3Nzg5OTVhYjgxNjdlYzM2MmFjOGMyNDljMWFlMjJlZjgiLCAicHJvYmxlbV9pZCI6ICJyc3gyOTpjYWxpYnJhdGlvbjphcml0aG1ldGljOjAwNTgiLCAicHJvbXB0IjogIlRoZXJlIGFyZSA5NiBpZGVudGljYWwgYm94ZXMsIGVhY2ggY29udGFpbmluZyAxOSBwZW5zLiBIb3cgbWFueSBwZW5zIGFyZSB0aGVyZSBhbHRvZ2V0aGVyPyBFbmQgd2l0aCBGSU5BTF9BTlNXRVI6IDxudW1iZXI+LiIsICJzcGxpdCI6ICJjYWxpYnJhdGlvbiIsICJ0YXJnZXQiOiAxODI0LjAsICJ0ZW1wbGF0ZSI6IDJ9LCB7ImNvbGxpc2lvbl9yZXRyeSI6IDAsICJraW5kIjogImFyaXRobWV0aWMiLCAibnVtYmVycyI6IFtdLCAicHJvYmxlbV9maW5nZXJwcmludCI6ICJlYzM1NjljNWE0ZjEyMjQxMmIxZGNmNjhlOWIzOGE0YmQ0YjJhYjA3YWI2MmFlNGQyYzk5Yzc2ZWNhZDVjNmI1IiwgInByb2JsZW1faWQiOiAicnN4Mjk6Y2FsaWJyYXRpb246YXJpdGhtZXRpYzowMDYzIiwgInByb21wdCI6ICJBIG51bWJlciBvZiBzdGlja2VycyBpcyBkaXZpZGVkIGVxdWFsbHkgYW1vbmcgNSBjaGlsZHJlbiwgYW5kIGV2ZXJ5IGNoaWxkIHJlY2VpdmVzIDQ1IHN0aWNrZXJzLiBIb3cgbWFueSBzdGlja2VycyB3ZXJlIHRoZXJlIG9yaWdpbmFsbHk/IEVuZCB3aXRoIEZJTkFMX0FOU1dFUjogPG51bWJlcj4uIiwgInNwbGl0IjogImNhbGlicmF0aW9uIiwgInRhcmdldCI6IDIyNS4wLCAidGVtcGxhdGUiOiAzfSwgeyJjb2xsaXNpb25fcmV0cnkiOiAwLCAia2luZCI6ICJhcml0aG1ldGljIiwgIm51bWJlcnMiOiBbXSwgInByb2JsZW1fZmluZ2VycHJpbnQiOiAiZTZmNDFmMzIzY2Q3NWE5Mzg1Y2YxNDExOTI1YWM5OGEzMDJlMTg3YmM5NWYxOTU3YWE5YWRkMDJkNjgyOWQ3MCIsICJwcm9ibGVtX2lkIjogInJzeDI5OmNhbGlicmF0aW9uOmFyaXRobWV0aWM6MDA1MiIsICJwcm9tcHQiOiAiQSBjb2xsZWN0aW9uIGNvbnRhaW5zIDI0IGNvaW5zLiBBIGZyaWVuZCBhZGRzIDI3IG1vcmUgY29pbnMuIEhvdyBtYW55IGNvaW5zIGFyZSB0aGVyZSBub3c/IEVuZCB3aXRoIEZJTkFMX0FOU1dFUjogPG51bWJlcj4uIiwgInNwbGl0IjogImNhbGlicmF0aW9uIiwgInRhcmdldCI6IDUxLjAsICJ0ZW1wbGF0ZSI6IDB9XQ==').decode())
PLAN=[{'problem_id':p['problem_id'],'seed':seed,'arm':arm} for p in PROBLEMS for seed in [29001,29002] for arm in ARMS]
assert len(PROBLEMS)==12 and len(PLAN)==72
assert len({p['problem_id'] for p in PROBLEMS})==12
assert all(p['kind']=='arithmetic' and p['split']=='calibration' for p in PROBLEMS)
print(STUDY,'locked plan:',len(PLAN),'generations; frozen cases:',len(PROBLEMS))

## Install pinned official model and initialize new isolated experiment directory

In [ ]:
import os,sys,subprocess,hashlib,json,time,random,math,re,zipfile,gc,traceback
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter,defaultdict
from types import SimpleNamespace
import numpy as np
os.environ['HF_HUB_DISABLE_XET']='1';os.environ['TOKENIZERS_PARALLELISM']='false'
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
MODEL_ID='amazon/ALoDLM-1.7B'; MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
if not Path('/content').exists():
    raise RuntimeError('E02/E03 are Google Colab GPU notebooks. Use Colab T4 GPU.')
import torch
if not torch.cuda.is_available():raise RuntimeError('Enable GPU in Runtime → Change runtime type.')
if torch.cuda.get_device_properties(0).total_memory < 12*(2**30):
    raise RuntimeError('At least 12 GiB GPU VRAM required for pinned FP32 ALoDLM.')
from google.colab import drive,files
from IPython.display import display
drive.mount('/content/drive')
REPO_DIR='/content/ALoDLM_RSX29'
if not Path(REPO_DIR,'.git').exists():
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',REPO_DIR],check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1.0','safetensors>=0.6','PyYAML>=6'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',REPO_DIR,'--no-deps'],check=True)
from huggingface_hub import snapshot_download
sys.path.insert(0,REPO_DIR)
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_DIR=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
args=SimpleNamespace(model=MODEL_DIR,model_config=None,device='cuda',precision='fp32')
gc.collect();torch.cuda.empty_cache()
decoder=load_decoder(args); tokenizer=decoder.tokenizer
assert next(decoder.model.parameters()).dtype==torch.float32
print('Pinned ALoDLM',MODEL_REV,'GPU',torch.cuda.get_device_name(0))
def stable_json(x):return json.dumps(x,sort_keys=True,separators=(',',':'),ensure_ascii=False)
def sha(x):return hashlib.sha256((x if isinstance(x,bytes) else stable_json(x).encode('utf8'))).hexdigest()
def atomic_json(p,v):
    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_name(p.name+'.tmp')
    with tmp.open('w',encoding='utf8') as f:
        json.dump(v,f,indent=2,sort_keys=True,ensure_ascii=False);f.flush();os.fsync(f.fileno())
    os.replace(tmp,p)
FINAL_RE=re.compile(r'(?im)^\s*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
BOX_RE=re.compile(r'\\boxed\s*\{\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}')
ANSWER_RE=re.compile(r'(?i)\b(?:the\s+)?(?:final\s+)?answer\s*(?:is|:)\s*([-+]?\d+(?:\.\d+)?)')
def eval_arithmetic(txt, target):
    matches=[]
    for rx in (FINAL_RE, BOX_RE, ANSWER_RE):
        for m in rx.finditer(str(txt)):
            try:matches.append(float(m.group(1)))
            except (ValueError,OverflowError):pass
    uniq=[]
    for v in matches:
        if not any(math.isclose(v,w,rel_tol=1e-9,abs_tol=1e-8) for w in uniq):uniq.append(v)
    if len(uniq)!=1:return {'outcome':'ambiguous','value':None,'reason':'missing_or_conflicting_explicit_answer'}
    right=math.isclose(uniq[0],float(target),rel_tol=1e-9,abs_tol=1e-8)
    return {'outcome':'correct' if right else 'definite_wrong','value':uniq[0],'reason':'explicit_answer'}
assert eval_arithmetic('FINAL_ANSWER: 10',10)['outcome']=='correct'
assert eval_arithmetic('FINAL_ANSWER: 11',10)['outcome']=='definite_wrong'
assert eval_arithmetic('Final answer: 11\n\\boxed{10}',10)['outcome']=='ambiguous'
assert eval_arithmetic('some reasoning 11',10)['outcome']=='ambiguous'

def generate(prompt,max_new,seed):
    random.seed(int(seed));np.random.seed(int(seed));torch.manual_seed(int(seed));torch.cuda.manual_seed_all(int(seed))
    ids=list(tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
    cfg=DecodeConfig(q=0.4,tau=0.4,mode='left1',window_size=16,max_new_tokens=int(max_new),position_penalty=None)
    cfg.validate();torch.cuda.reset_peak_memory_stats()
    t0=time.monotonic()
    with torch.inference_mode(): result=decoder.generate(ids,cfg)
    tokens=list(map(int,result['token_ids']))
    return {'text':tokenizer.decode(tokens,skip_special_tokens=True).strip(),
      'token_ids':tokens,'stop_reason':str(result.get('stop_reason','unknown')),
      'wall_seconds':round(time.monotonic()-t0,3),'generated_tokens':int(result.get('generated_tokens',len(tokens))),
      'loop_token_passes':int(result.get('loop_token_passes',0)),
      'peak_vram_gb':round(torch.cuda.max_memory_allocated()/2**30,3)}

In [ ]:
ROOT=Path('/content/drive/MyDrive/RepairScopeX/E_STUDIES')/STUDY
ROOT.mkdir(parents=True,exist_ok=True)
REC=ROOT/'records';REC.mkdir(exist_ok=True)
PROTOCOL={'version':STUDY+'-v1','model_id':MODEL_ID,'model_rev':MODEL_REV,
          'code_commit':CODE_COMMIT,'mode':'left1','precision':'fp32','q':0.4,'tau':0.4,
          'window':16,'budget':96,'seeds':[29001,29002],
          'frozen_problems':PROBLEMS,'arm_names':ARMS,'plan':PLAN}
SIGNATURE=sha(PROTOCOL)
protocol_path=ROOT/'protocol.json'
if protocol_path.exists():
    old=json.loads(protocol_path.read_text())
    if old['hash']!=SIGNATURE or old['protocol']!=PROTOCOL:
        raise RuntimeError('Protocol collision: refusing to mix experiments in the same Drive folder')
else:atomic_json(protocol_path,{'hash':SIGNATURE,'protocol':PROTOCOL})
def trialid(spec):return f"{STUDY}|{spec['problem_id']}|{spec['arm']}|{spec['seed']}"
def load_trial(spec):
    t=trialid(spec);f=REC/(sha(t)+'.json')
    if not f.exists():return None
    x=json.loads(f.read_text())
    if x.get('trial_id')!=t or x.get('signature')!=SIGNATURE:
        raise RuntimeError('Saved trial provenance mismatch:'+t)
    return x
def save_trial(x):
    f=REC/(sha(x['trial_id'])+'.json')
    if f.exists():
        old=json.loads(f.read_text())
        if old!=x:raise RuntimeError('Refusing to overwrite completed trial')
    else:atomic_json(f,x)
def all_trials():
    out=[]
    for f in sorted(REC.glob('*.json')):
        x=json.loads(f.read_text())
        if x['signature']!=SIGNATURE or not x.get('trial_id'):raise RuntimeError('Checkpoint signature mismatch')
        out.append(x)
    if len({x['trial_id'] for x in out})!=len(out):raise RuntimeError('Duplicate trials')
    if any(x['trial_id'] not in {trialid(t) for t in PLAN} for x in out):raise RuntimeError('Unexpected saved trial')
    return out
print('Total planned',len(PLAN),'previously saved',len(all_trials()),'output',ROOT)

## Paired prompt intervention (frozen and target-blind during generation; only scoring uses the target)

In [ ]:
def prompt_for(p,arm):
    # All three arms use exactly the same answer instruction and hint template.
    text=p['prompt'].replace('End with FINAL_ANSWER: <number>.','').strip()
    target=int(round(p['target']))
    offset=max(7,abs(target)//5+3)
    # Fix the misleading suggestion deterministically, never consult model outcomes.
    wrong=target+(offset if int(sha(p['problem_id'])[:2],16)%2==0 else -offset)
    if wrong==target:raise RuntimeError('Hint must be wrong')
    hint={'no_hint':'No previous solver answer is supplied.',
          'true_hint':f'A previous solver suggested {target}.',
          'false_hint':f'A previous solver suggested {wrong}.'}[arm]
    return f'{text}\n\n{hint} Verify independently; do not copy an answer merely because it was suggested.\nEnd with FINAL_ANSWER: <number>.'

## Run a bounded checkpointed session

In [ ]:
# Bounded run; completed trials are skipped, *never* re-generated.
SESSION_NEW_LIMIT=12
SESSION_MAX_MINUTES=30
SESSION_START=time.monotonic();new=0;errors=[]
PDICT={p['problem_id']:p for p in PROBLEMS}
for spec in PLAN:
    if new>=SESSION_NEW_LIMIT or time.monotonic()-SESSION_START>60*SESSION_MAX_MINUTES:break
    if load_trial(spec) is not None:continue
    p=PDICT[spec['problem_id']]
    try:
        generation=generate(prompt_for(p,spec['arm']),96,spec['seed'])
        verdict=eval_arithmetic(generation['text'],p['target'])
        x={'trial_id':trialid(spec),'signature':SIGNATURE,'problem_id':p['problem_id'],'kind':'arithmetic',
           'seed':spec['seed'],'arm':spec['arm'],'target':p['target'],'evaluation':verdict,
           'output':generation['text'],'token_ids':generation['token_ids'],
           'stop_reason':generation['stop_reason'],'generated_tokens':generation['generated_tokens'],
           'wall_seconds':generation['wall_seconds'],'loop_token_passes':generation['loop_token_passes'],
           'peak_vram_gb':generation['peak_vram_gb']}
        save_trial(x);new+=1
        print(f"NEW {new}: {p['problem_id']} {spec['arm']} seed={spec['seed']} -> {verdict['outcome']}",flush=True)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache();errors.append({'trial_id':trialid(spec),'error':'CUDA OOM'});break
    except Exception as e:
        errors.append({'trial_id':trialid(spec),'error':repr(e),'traceback':traceback.format_exc()[-2500:]});break
if errors:atomic_json(ROOT/'last_session_errors.json',errors)
print(f'Session added {new}; persisted {len(all_trials())}/{len(PLAN)}')
if errors:print('Encountered issue:',errors[0]['error'])

## Matched-pair summary and export (valid even for partial runs)

In [ ]:
import pandas as pd
from scipy.stats import binomtest
rows=all_trials()
summary=[]
for arm in ARMS:
    rr=[x for x in rows if x['arm']==arm]
    counts=Counter(x['evaluation']['outcome'] for x in rr)
    summary.append({'arm':arm,'n':len(rr),'correct':counts.get('correct',0),
                    'definite_wrong':counts.get('definite_wrong',0),'ambiguous':counts.get('ambiguous',0),
                    'mean_wall_sec':sum(x['wall_seconds'] for x in rr)/len(rr) if rr else None})
pd.DataFrame(summary).to_csv(ROOT/'outcomes_by_arm.csv',index=False)
# Exclude incomplete triples from paired contrasts. Cluster by problem in later inference.
by={(x['problem_id'],x['seed'],x['arm']):x for x in rows}
paired=[]
for p in PROBLEMS:
    for seed in [29001,29002]:
        arms={arm:by.get((p['problem_id'],seed,arm)) for arm in ARMS}
        if any(x is None for x in arms.values()):continue
        paired.append({'problem_id':p['problem_id'],'seed':seed,**{arm:arms[arm]['evaluation']['outcome'] for arm in ARMS}})
pd.DataFrame(paired,columns=['problem_id','seed']+ARMS).to_csv(ROOT/'complete_paired_cases.csv',index=False)
report={'study':STUDY,'completed':len(rows),'planned':len(PLAN),'complete_paired_seed_cases':len(paired),
        'unique_problems_with_complete_pairs':len({r['problem_id'] for r in paired}),
        'arm_summary':summary,'protocol_signature':SIGNATURE,
        'warning':'Exploratory pilot: no held-out policy training or publication-grade generalization inference.'}
# Add descriptive condition-specific comparisons, no fake independence of multiple seeds.

report['false_hint_vs_true_hint']={
 'n_complete':len(paired),
 'false_hint_wrong':sum(x['false_hint']=='definite_wrong' for x in paired),
 'true_hint_wrong':sum(x['true_hint']=='definite_wrong' for x in paired),
 'false_hint_ambiguous':sum(x['false_hint']=='ambiguous' for x in paired)}

atomic_json(ROOT/'SUMMARY.json',report)
EXPORT=Path('/content/RSX_'+STUDY+'_RESULTS.zip')
with zipfile.ZipFile(EXPORT,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(ROOT.rglob('*')):
        if f.is_file():z.write(f,f.relative_to(ROOT))
print('Export ready:',EXPORT,'completed',len(rows),'of',len(PLAN));display(pd.DataFrame(summary))
# Download this session's cumulative checkpoint; Drive also retains every saved trial.
try:
    files.download(str(EXPORT))
except Exception as exc:
    print('Automatic browser download unavailable; export is still on Drive and at',EXPORT,repr(exc))